# Lie symmetry methods for PDEs with `liepde`

This notebook gives some examples demonstrating use of `liepde(...)`. By default, `liepde(...)` returns the best readable solution it can find. Use `result_level="reduction"` to ask for the best reduction-level object, and `return_details=True` or `result_level="details"` to inspect the full solver record. Besides the examples below, the package includes `benchmarks/benchmark_common_pdes.py`, which exercises about 20 common PDE examples and prints both per-case status and a solved/reduced/failed summary.

## Imports


In [ ]:
import sympy as sp
import liepde as lp

x, t, y = sp.symbols("x t y", positive=True)
u = sp.Function("u")

## Examples

The examples below use the current high-level API on several standard PDEs, including the transport equation, the heat equation, linear reaction-diffusion, and Burgers-type equations.

## Transport equation

For $u_t + u_x = 0$, the package returns the characteristic-form general solution by default.


In [2]:
transport_eq = sp.Eq(u(x, t).diff(t) + u(x, t).diff(x), 0)
transport_sol = lp.liepde(transport_eq, u, (x, t))
transport_sol


Eq(u(x, t), F(-t + x))

In [ ]:
transport_details = lp.liepde(transport_eq, u, (x, t), return_details=True)
{
    "basis_vector_count": len(transport_details.basis_vectors),
    "warnings": transport_details.diagnostics.warnings,
}

{'basis_vector_count': 13,
 'warnings': ('reduction search produced a non-usable reduced equation; discarding reduction result.',)}

## Heat equation

The main reduction pipeline still often falls back to a known similarity solution for the heat equation. The default API returns the readable solution; the details object records the fallback path.


In [ ]:
heat_eq = sp.Eq(u(x, t).diff(t) - u(x, t).diff(x, 2), 0)
heat_sol = lp.liepde(heat_eq, u, (x, t))
heat_sol

Eq(u(x, t), C1 + C2*erf(x/(2*sqrt(t))))

In [ ]:
heat_details = lp.liepde(heat_eq, u, (x, t), return_details=True)
{
    "direct_solution": heat_details.direct_solution,
    "warnings": heat_details.diagnostics.warnings,
}

{'direct_solution': Eq(u(x, t), C1 + C2*erf(x/(2*sqrt(t)))),
 'warnings': ('reduction search produced a non-usable reduced equation; discarding reduction result.',
  'direct pdsolve fallback failed: NotImplementedError: psolve: Cannot solve Derivative(u(x, t), t) - Derivative(u(x, t), (x, 2))',
  'used known-solution fallback: _reaction_diffusion_similarity_solution.')}

## Linear reaction-diffusion equation

The current package also includes a small known-solution fallback for linear reaction-diffusion/advection-diffusion-reaction equations of the form $u_t + a u_x - D u_{xx} - \lambda u = 0$.


In [ ]:
reaction_eq = sp.Eq(u(x, t).diff(t) - u(x, t).diff(x, 2) - u(x, t), 0)
reaction_sol = lp.liepde(reaction_eq, u, (x, t))
reaction_sol

Eq(u(x, t), (C1 + C2*erf(x/(2*sqrt(t))))*exp(t))

## Reduction-level and failure-mode controls

Use `result_level="reduction"` and `failure_mode="status"` when you want a clean failure object instead of bare `None`.


In [ ]:
advection_2d_eq = sp.Eq(u(x, y, t).diff(t) + u(x, y, t).diff(x) + u(x, y, t).diff(y), 0)
advection_2d_result = lp.liepde(
    advection_2d_eq,
    u,
    (x, y, t),
    result_level="reduction",
    failure_mode="status",
)
advection_2d_result

LiePDEFailure(reason='direct pdsolve fallback failed: NotImplementedError: Right now only partial differential equations of two variables are supported', warnings=('degree 1 reduction search failed: ValueError: Invalid coordinate chart: expected 3 total coordinates for variables (x, y, t), got 2 (invariants=1, transverse=1).', 'direct pdsolve fallback failed: NotImplementedError: Right now only partial differential equations of two variables are supported'))

## Analysis-only and classification helpers


In [ ]:
lp.classify_pde(heat_eq, u, (x, t))

LiePDEAnalysis(order=2, principal_symbol=u_t, solved_rhs=u_x2, indep_vars=(x, t), dependent=u(x, t), is_linear=True, is_homogeneous=True)

In [ ]:
lp.liepde(heat_eq, u, (x, t), analyze_only=True)

LiePDEAnalysis(order=2, principal_symbol=u_t, solved_rhs=u_x2, indep_vars=(x, t), dependent=u(x, t), is_linear=True, is_homogeneous=True)

## Burgers example

For the viscous Burgers equation, the current package often returns a simple constant solution family rather than a richer transformed solution.

In [10]:
burgers_eq = sp.Eq(u(x, t).diff(t) + u(x, t) * u(x, t).diff(x) - u(x, t).diff(x, 2), 0)
lp.liepde(burgers_eq, u, (x, t))

Eq(u(x, t), C1)

## Symmetry Types

Common symmetry types in the analysis of differential equations include:

- **point symmetries** — transformations of the independent and dependent variables
- **contact symmetries** — transformations that may also depend on first derivatives
- **generalized (Lie–Bäcklund) symmetries** — transformations that may depend on higher derivatives
- **variational symmetries** — symmetries of an action functional, related to Noether’s theorem
- **divergence / Noether-type symmetries** — symmetries preserving a Lagrangian up to a total divergence
- **potential symmetries** — symmetries that appear after introducing potential variables from conservation laws
- **hidden symmetries** — symmetries that become visible only after reduction or transformation
- **nonclassical / conditional symmetries** — symmetries of the PDE together with an invariant-surface condition
- **approximate symmetries** — symmetries valid to a given order in a perturbation parameter
- **discrete symmetries** — finite symmetries such as reflections or permutations

## Classical vs. Nonclassical Symmetry Methods

In symmetry analysis of differential equations, “symmetry” can mean several different things depending on what is allowed to transform and how strongly the transformed object must preserve the equation.

### Main types of symmetries

#### Point symmetries

These are the standard Lie symmetries used in most introductory symmetry reduction methods. They transform only the independent and dependent variables, for example

$$
(x,t,u) \mapsto (\tilde x,\tilde t,\tilde u),
$$

with infinitesimal generator

$$
X=\sum_i \xi_i(x,u)\,\partial_{x_i}+\sum_\alpha \phi_\alpha(x,u)\,\partial_{u^\alpha}.
$$

They do **not** depend explicitly on derivatives such as $u_x$ or $u_{xx}$. They are called “point” symmetries because they act on points of the space of variables $(x,u)$, and are then prolonged to jet space.

Typical examples include:
- translations in $x$ or $t$
- scalings
- Galilean boosts
- rotations

#### Contact symmetries

These are more general than point symmetries. They may depend on first derivatives while preserving the contact structure of first-order jet space. Contact symmetries are especially important for first-order PDEs and Hamilton–Jacobi-type equations.

#### Generalized symmetries / Lie–Bäcklund symmetries

These are broader still. Their infinitesimals may depend on derivatives of arbitrary order:

$$
X=\sum_i \xi_i(x,u^{(r)})\,\partial_{x_i}+\sum_\alpha \phi_\alpha(x,u^{(r)})\,\partial_{u^\alpha}.
$$

In evolutionary form, one often works with a characteristic $Q(x,u^{(r)})$. These symmetries are especially important in integrable systems, where higher-order symmetry hierarchies often appear.

#### Variational symmetries

These are symmetries of an **action functional**, not just of the differential equation. Their importance comes from **Noether’s theorem**: under suitable assumptions, each variational symmetry yields a conservation law for the Euler–Lagrange equations.

#### Divergence / Noether-type symmetries

These preserve a Lagrangian up to a total divergence rather than exactly. They still generate conservation laws through Noether’s theorem, and often arise naturally in physical applications.

#### Potential symmetries

These arise after introducing **potential variables** from a conservation law. A PDE may have no useful point symmetry in its original variables, but after rewriting it as a potential system, new symmetries may appear. These are called potential symmetries.

#### Hidden symmetries

A hidden symmetry is not visible in the original equation in the symmetry class being considered, but appears after reduction or transformation. Such symmetries can often be used for additional reduction or integration.

#### Nonclassical / conditional symmetries

These are not symmetries of the original PDE alone, but of the PDE together with an additional invariant-surface condition. They often yield reductions that are not obtainable from the classical point-symmetry algebra.

---

### “Classical” symmetry methods

The **classical Lie method** usually means the standard point-symmetry method:

1. choose a point-symmetry generator
   $$
   X=\sum_i \xi_i(x,u)\partial_{x_i}+\sum_\alpha \phi_\alpha(x,u)\partial_{u^\alpha},
   $$
2. prolong it to derivatives,
3. impose the infinitesimal invariance condition on the PDE,
4. solve the determining equations for $\xi_i,\phi_\alpha$,
5. use the resulting symmetry to find invariants and reduce the PDE.

The key point is that the invariance condition is imposed on the **solution manifold of the PDE alone**.

---

### “Nonclassical” symmetry methods

The **nonclassical method** modifies the invariance problem by adding the **invariant-surface condition**

$$
Q := \phi - \sum_i \xi_i u_i = 0
$$

for scalar PDEs, or its system analogue.

One then demands invariance not of the PDE alone, but of the enlarged system:
- the original PDE
- the invariant-surface condition

So the nonclassical method looks for vector fields whose invariant solutions satisfy both the PDE and the side condition determined by the symmetry itself.

---

### The core difference between classical and nonclassical methods

The difference is not just a matter of using more complicated formulas. It is a different invariance concept.

#### Classical method

One requires

$$
\operatorname{pr}^{(n)}X(F)\Big|_{F=0}=0,
$$

where $F=0$ is the PDE.

So the vector field must generate a symmetry of the whole equation.

#### Nonclassical method

One requires

$$
\operatorname{pr}^{(n)}X(F)\Big|_{F=0,\;Q=0,\;\text{differential consequences of }Q=0}=0.
$$

So the vector field only needs to preserve the PDE on the subset of solutions that are also invariant under that vector field.

This is why the nonclassical method can produce reductions that the classical method misses.

---

### Why nonclassical symmetries can find more reductions

Classical symmetries describe transformations that map **solutions to solutions** in the usual global symmetry sense.

Nonclassical symmetries instead target **special invariant families of solutions**. They do not need to preserve the entire solution set in the same strong way. Because of that, they can reveal useful ansätze and reductions that do not arise from the classical point-symmetry algebra.

So, roughly speaking:

- **classical symmetries** describe the global symmetry structure of the PDE
- **nonclassical symmetries** provide a broader reduction mechanism for special invariant solutions

---

### Practical differences

#### Classical method

Advantages:
- determining equations are linear in the infinitesimals
- theory is well developed
- symmetry algebra structure is often easy to interpret
- comparatively straightforward to automate

Disadvantages:
- may miss many useful reductions
- often finds only the most standard invariant solution families

#### Nonclassical method

Advantages:
- can produce reductions unavailable by the classical method
- often yields more specialized and interesting exact solutions

Disadvantages:
- determining equations become nonlinear and much harder
- redundancy and equivalence issues are more severe
- automation is substantially harder
- the geometric interpretation is more subtle

---

### A useful intuition

Suppose one is looking for similarity solutions of the form

$$
u(x,t)=f\!\left(\frac{x}{\sqrt{t}}\right).
$$

In the classical method, one first identifies an actual symmetry of the PDE, such as a scaling symmetry, and then derives the invariant variable from that symmetry.

In the nonclassical method, one can sometimes impose a compatible invariant-surface condition directly and obtain a reduction even when no corresponding classical point symmetry produces it in the standard way.

So the contrast is:

- **Classical method:**: This transformation is a genuine symmetry of the PDE.
- **Nonclassical method:**: This vector field defines a compatible invariant ansatz for a special family of solutions.

---

### Relationship among the main symmetry notions

A useful rough hierarchy is:

- **point symmetries**: depend only on $x,u$
- **contact symmetries**: may depend on first derivatives
- **generalized symmetries**: may depend on higher derivatives
- **variational symmetries**: symmetries of an action functional
- **potential symmetries**: symmetries after introducing potential variables
- **nonclassical symmetries**: symmetries of the PDE together with an invariant-surface condition

These are not all nested in one simple chain, but point/contact/generalized are often viewed as progressively broader transformation classes, while variational, potential, and nonclassical refer to different structural settings.

---

### For a Lie-symmetry PDE package

For a package focused on Lie symmetry reduction, the most important distinctions are usually:

- **classical point symmetries** — the natural foundation for standard symmetry reduction
- **nonclassical symmetries** — a major extension for finding additional reductions
- **potential symmetries** — especially valuable once conservation-law machinery is available
- **generalized symmetries** — a larger long-term extension beyond point methods

A concise summary is:

- **Classical Lie symmetry method:** find point symmetries of the PDE itself and reduce using their invariants.
- **Nonclassical symmetry method:** augment the PDE with the invariant-surface condition and search for compatible invariant reductions, often yielding solution families missed by the classical method.

## References

- Bluman, G. W., and Cole, J. D. 1969. *The General Similarity Solution of the Heat Equation*. Journal of Mathematics and Mechanics 18(11): 1025–1042.
- Bluman, G. W., and Kumei, S. 1989. *Symmetries and Differential Equations*. Springer.
- Ibragimov, N. H. 1994. *CRC Handbook of Lie Group Analysis of Differential Equations, Vol. 1*. CRC Press.
- Olver, P. J. 1993. *Applications of Lie Groups to Differential Equations*. 2nd ed. Springer.
- Ovsiannikov, L. V. 1982. *Group Analysis of Differential Equations*. Academic Press.